In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
df_dc = pd.read_csv("../data/raw/wikipedia_jogos_dc.csv")
df_marvel = pd.read_csv("../data/raw/wikipedia_jogos_marvel.csv")

print(df_dc.columns.tolist())
print(df_marvel.columns.tolist())

In [ ]:
mapa_dc = {"Title": "titulo", "Year": "ano", "Platform(s)": "plataformas", "Publisher(s)": "publicadora(s)"}
mapa_marvel = {"Título": "titulo", "Data": "ano", "Plataforma(s)": "plataformas", "Publicadora(s)": "publicadora(s)"}

colunas_finais = ["titulo", "ano", "plataformas", "publicadora(s)", "universo", "secao_wikipedia", "url_fonte", "data_coleta"]

df_dc = df_dc.rename(columns=mapa_dc)
df_dc["universo"] = "DC"

df_marvel = df_marvel.rename(columns=mapa_marvel)
df_marvel["universo"] = "Marvel"

df_jogos = pd.concat([df_dc[colunas_finais], df_marvel[colunas_finais]], ignore_index=True)
print(df_jogos.shape)

In [ ]:
for col in ["titulo", "ano", "plataformas", "publicadora(s)"]:
    df_jogos[col] = (df_jogos[col].astype("string")
                     .str.replace(r"\[.*?\]", "", regex=True)
                     .str.strip())

In [ ]:
ano_num = pd.to_numeric(df_jogos["ano"], errors="coerce")
print(df_jogos.loc[ano_num.isna() & df_jogos["ano"].notna(), "ano"].value_counts())
#df_jogos[df_jogos["ano"] == "TBA"]
#df_jogos[df_jogos["ano"] == "2011, 2013"]
#df_jogos[df_jogos["ano"] == "Upcoming"]
#df_jogos[df_jogos["ano"] == "Ainda não lançado"]

#Verifica possiveis duplicatas
mascara = df_jogos.duplicated(subset=["titulo", "universo"], keep=False)
duplicados = df_jogos[mascara].sort_values(["universo", "titulo"])

print(len(duplicados))
duplicados

#Verifica numero de duplicatas reais
cols_conteudo = ["titulo", "ano", "plataformas", "publicadora(s)", "universo"]
df_jogos.duplicated(subset=cols_conteudo).sum()

In [ ]:
#Remove linhas N/A coletadas por engano
vazias = df_jogos[df_jogos["titulo"].isna()]
print(len(vazias))

df_jogos = df_jogos.dropna(subset=["titulo"]).reset_index(drop=True)

#Remove jogos ainda não lançados/com dados inexistentes ou ausentes.
valores_remover = ["TBA", "Upcoming", "Ainda não lançado"]

antes = len(df_jogos)
df_jogos = df_jogos[~df_jogos["ano"].isin(valores_remover)].copy()
df_jogos = df_jogos.reset_index(drop=True)
print(f"Linhas removidas: {antes - len(df_jogos)}")


#Cria uma nova coluna com o ano de lançamento do jogo.
df_jogos["ano_lancamento"] = (df_jogos["ano"]
                              .str.extract(r"(\d{4})", expand=False)
                              .astype("Int64"))

In [ ]:
pasta = Path("../data/processed")
pasta.mkdir(parents=True, exist_ok=True)
df_jogos.to_csv(pasta / "jogos_dc_marvel.csv", index=False, encoding="utf-8")